## process and functions to derive the normalisation parameters to be used, from the suite of test sites and other reference data
### Steps are:
#### 1. Read in the summary statistics file produced by running the water quality process over all the dev locations 
- this file is 
#### 2. 

In [34]:
def part9a_normalise_wq_stats(wq_allstats,filename,verbose=False,test=False):
    #--- this function determines the scale and offset needed to re-scale like measures 
    #--- two 'natural' assumptions are involved:
    #----- 1) that there is a natural zero, and 
    #----- 2) that median values of all measures will, over a large number of samples, converge 
    # the target variable is taken to be the median for convergence, but where there is an observed continental median available, it makes 
    # more sense to use that - e.g., for chla., 24 mg/l approx. 

    #choose target variable for normalisation - the variable that seems most realistic / stable / etc. 
    tss_target  = 'tsm_lym_oli_agm'        #this has overall q10 = 16ppt, median =  36 ppt, q90 =  59 ppt
                                           #over all measures q10 = 32, median = 44, q90 = 53. 
    chla_target = 'chla_meris2b_msi_agm'   #this seems less sensitive than the MODIS two-band model
    chla_target = 'chla_modis2b_msi_agm'   
    continental_average = True             #this seems to be a better overall approach than choosing one model as best
    if continental_average : chla_target_median = 24.5; chla_target_low = 0.0  #continental median value
    
    Q_tss  = 0.1     #the quantile to use when matching low values; this is an important parameter. Choose to match zero of the target variable.
    Q_chla = 0.1
    #initialise a dataset to hold the normalisation parameters
    np_data = wq_allstats.drop_dims(('place'))

    if test or verbose : print('Calculating scale and offset for data normalisation..')
    #--- run through all the wq measures 
    for varname in list(wq_allstats.data_vars):
        if varname.endswith('_med') : 
            varname_base =  varname[:varname.find('_med')]
            if 'tss'  in wq_allstats[varname].dims : 
                targetvar = tss_target;  Q = Q_tss
                scale       = np.round((wq_allstats[   targetvar+'_med'].median(dim=('place')) - wq_allstats[   targetvar+'_low'].quantile(Q,dim=('place'))) /\
                                       (wq_allstats[varname_base+'_med'].median(dim=('place')) - wq_allstats[varname_base+'_low'].quantile(Q,dim=('place'))),3)          
                offset      = np.round (wq_allstats[    targetvar+'_low'].quantile(Q,dim=('place')) - ( wq_allstats[varname_base+'_low'].quantile(Q,dim=('place')) * scale ),3) 
            
            if 'chla' in wq_allstats[varname].dims : 
                targetvar = chla_target; Q = Q_chla
                if continental_average:
                    scale       = np.round((chla_target_median - chla_target_low) /\
                                           (wq_allstats[varname_base+'_med'].median(dim=('place')) - wq_allstats[varname_base+'_low'].quantile(Q,dim=('place'))),3)          
                    offset      = np.round ((chla_target_low)                                      - (wq_allstats[varname_base+'_low'].quantile(Q,dim=('place')) * scale ),3) 
                else :
                    scale       = np.round((wq_allstats[   targetvar+'_med'].median(dim=('place')) - wq_allstats[   targetvar+'_low'].quantile(Q,dim=('place'))) /\
                                       (wq_allstats[varname_base+'_med'].median(dim=('place')) - wq_allstats[varname_base+'_low'].quantile(Q,dim=('place'))),3)          
                    offset      = np.round (wq_allstats[    targetvar+'_low'].quantile(Q,dim=('place')) - ( wq_allstats[varname_base+'_low'].quantile(Q,dim=('place')) * scale ),3) 
            if test or verbose : print(varname_base, scale.values, offset.values)
            np_data[varname_base+'_scale' ] = scale;
            np_data[varname_base+'_offset'] = offset;
    
    np_data.to_netcdf(filename)        

    # --- convert the normalisation parameter values to a dictionary for simpler handling ---
    np_data_dict = {}

    # --- for varname in (['ndssi_rg_msi_agm_scale','ndssi_rg_msi_agm_offset']):
    for varname in (np_data.data_vars):
        value   = np_data[varname].item(0)  
        if varname.endswith('_scale'):
            base = varname[:varname.find('_scale')]
            if not base in np_data_dict.keys():
                np_data_dict[base] = {}
            np_data_dict[base]['scale'] = value
        if varname.endswith('_offset'):
            base = varname[:varname.find('_offset')]
            if not base in np_data_dict.keys():
                np_data_dict[base] = {}
            np_data_dict[base]['offset'] = value

    # --- write the dictonary to a file in the work area for WP 1.4 --- 
    jfilename = '../dev/deafrica_water_quality/WP1.4/'+nparams_filename[:nparams_filename.find('.')]+'.json'
    with open(jfilename, 'w') as f:
      json.dump(np_data_dict, f)

    #with open(jfilename) as f:
    #  np_data_dict1 = json.load(f)

    return np_data